In [1]:
import numpy as np
import cv2 as cv
import os
import pandas as pd
from tqdm import tqdm

In [2]:
RUTA_IMAGENES = 'data_proyecto_bigkk/train_images'
NOMBRE_CSV_SALIDA = 'features_imagenes.csv'

In [ ]:
def aplicar_sift(img_gray, ):
    if img_gray is None or len(img_gray.shape) != 2:
        raise ValueError("Debe proporcionar una imagen válida en escala de grises.")

    sift = cv.SIFT_create()

    keypoints, descriptors = sift.detectAndCompute(img_gray, None)
    #Generar un solo vector descriptor de 128
    descriptors = np.mean(descriptors,axis=0)

    return descriptors

In [4]:
mascotas_features_temp = {}

In [5]:
archivos = [f for f in os.listdir(RUTA_IMAGENES) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
for nombre_archivo in tqdm(archivos, desc="Procesando imágenes"):
    ruta_completa = os.path.join(RUTA_IMAGENES, nombre_archivo)

    pet_id = nombre_archivo.split('-')[0]

    img_gray = cv.imread(ruta_completa, 0)

    if img_gray is None:
        print(f"Advertencia: No se pudo leer la imagen {nombre_archivo}. Saltando.")
        continue

    vector_imagen = aplicar_sift(img_gray)

    if vector_imagen is None:
        continue

    if pet_id not in mascotas_features_temp:
        mascotas_features_temp[pet_id] = []

    mascotas_features_temp[pet_id].append(vector_imagen)


print("\nProcesamiento de imágenes terminado. Calculando promedios finales por mascota...")

Procesando imágenes: 100%|██████████| 58311/58311 [33:47<00:00, 28.75it/s]  


Procesamiento de imágenes terminado. Calculando promedios finales por mascota...


In [ ]:
lista_datos_finales = []

for pet_id, lista_vectores in mascotas_features_temp.items():
    vector_final_mascota = np.mean(lista_vectores, axis=0)

    lista_datos_finales.append({
        'PetID': pet_id,
        'FeatureVector': vector_final_mascota.tolist() 
    })

In [8]:
# Crear DataFrame de pandas
df_mascotas = pd.DataFrame(lista_datos_finales)

# Exportar a CSV
df_mascotas.to_csv(NOMBRE_CSV_SALIDA, index=False)

print(f"\n¡Éxito! Archivo CSV creado: {NOMBRE_CSV_SALIDA}")
print(f"Total de mascotas únicas procesadas: {len(df_mascotas)}")
print("Ejemplo de las primeras filas del DataFrame:")
print(df_mascotas.head())


¡Éxito! Archivo CSV creado: features_imagenes.csv
Total de mascotas únicas procesadas: 14652
Ejemplo de las primeras filas del DataFrame:
       PetID                                      FeatureVector
0  0008c5398  [26.99315071105957, 20.499935150146484, 16.190...
1  000a290e4  [23.275924682617188, 25.52924346923828, 25.638...
2  000fb9572  [26.8854923248291, 22.587305068969727, 16.1346...
3  0011d7c25  [23.846139907836914, 22.660888671875, 21.24773...
4  00156db4a  [25.454547882080078, 26.718135833740234, 24.28...


In [ ]:
ruta_train_csv = 'data_proyecto_bigkk/train/train.csv' 
df_train_master = pd.read_csv(ruta_train_csv)

df_todos_los_ids = df_train_master[['PetID']].copy()

df_final = pd.merge(df_todos_los_ids, df_mascotas, on='PetID', how='left')

print(f"Total de IDs en Train original: {len(df_todos_los_ids)}")
print(f"Total de filas en CSV final: {len(df_final)}")
print(f"Cantidad de mascotas sin fotos (Nulos): {df_final['FeatureVector'].isna().sum()}")

df_final.to_csv('features_mascotas_completo.csv', index=False)
print("Archivo 'features_mascotas_completo.csv' guardado exitosamente.")

Total de IDs en Train original: 14993
Total de filas en CSV final: 14993
Cantidad de mascotas sin fotos (Nulos): 341
Archivo 'features_mascotas_completo.csv' guardado exitosamente.
